# Official Replay Metric-Aligned Checkpoint/BC Screen v1

This notebook executes the next model-selection step from `walkforward_retention_next_training_handoff_v1`: aggregate existing official warm-start replay candidates, score them with one replay metric contract, and decide whether any candidate can be retained, used as a checkpoint/BC reference, or discarded.

Governance:
- Baselines: canonical reference `soxl_backtest_policy_reimplementation_v1`; direct comparison `walkforward_retention_next_training_handoff_v1` and strict retention gate `model_retention_walkforward_gate_v1`.
- Role: `diagnose` / `replace_selection`; no actor promotion is attempted here.
- Target windows: official warm-start `wf_2022`, `wf_2023_2024`; final retention still requires `wf_2020`, `wf_2022`, `wf_2023_2024` plus current strict return alignment.
- Accepted trade-off: a near-miss can become a checkpoint/BC teacher reference only; it cannot become a retained model until it clears full official replay and final walk-forward gates.
- Changed layer: `selection` and `evaluation`, not reward, network, or execution code.
- Closest historical success: short-volshock action-cap official replay near-miss (`cap05_hit2`).
- Closest historical failure: sequence/trend and state-conditioned BC screens improved some windows but failed official warm-start gates.
- Why not duplicate: this screen ranks candidates across multiple prior cards with a single official replay metric contract and attaches final-retention coverage blockers.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for cur in (start, *start.parents):
        if (cur / "AGENTS.md").exists() and (cur / "SPEC.md").exists():
            return cur
    raise FileNotFoundError("Could not find repo root containing AGENTS.md and SPEC.md")


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.official_candidate_aggregation import (
    OfficialReplayCandidateConfig,
    normalise_official_replay_candidate_frame,
    render_official_replay_candidate_review,
    score_official_replay_candidate_pool,
    score_official_replay_stress_scenarios,
)

CARD = ROOT / "runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1"
CARD.mkdir(parents=True, exist_ok=True)


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))


def load_json(rel_path: str):
    path = ROOT / rel_path
    if not path.exists():
        return None
    return json.loads(path.read_text())


def summary_of(obj):
    if isinstance(obj, dict) and isinstance(obj.get("summary"), dict):
        return obj["summary"]
    return obj if isinstance(obj, dict) else {}


def write_json(name: str, payload) -> str:
    path = CARD / name
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=True) + "\n")
    return rel(path)


def write_csv(name: str, frame: pd.DataFrame) -> str:
    path = CARD / name
    frame.to_csv(path, index=False)
    return rel(path)

CONFIG = OfficialReplayCandidateConfig(
    target_relative_return_pp=30.0,
    target_max_dd_pct=15.0,
    robust_relative_margin_pp=2.0,
    robust_max_dd_margin_pct=1.0,
    near_miss_relative_gap_pp=1.0,
    near_miss_max_dd_gap_pct=0.5,
)
FINAL_REQUIRED_WINDOWS = {"wf_2020", "wf_2022", "wf_2023_2024"}
EVAL_REQUIRED_WINDOWS = {"wf_2022", "wf_2023_2024"}

SOURCE_TABLE = [
    {
        "source_label": "sequence_trend_joint_warm_start_official_eval_v1",
        "mechanism": "sequence_trend_joint_bc",
        "metrics_path": "runtime/model_selection/card/sequence_trend_joint_warm_start_official_eval_v1/warm_start_official_metrics.csv",
    },
    {
        "source_label": "state_conditioned_multi_objective_bc_screen_v1",
        "mechanism": "state_conditioned_multi_objective_bc",
        "metrics_path": "runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1/warm_start_official_metrics.csv",
    },
    {
        "source_label": "short_volshock_bc_checkpoint_alignment_v1",
        "mechanism": "short_volshock_checkpoint_bc",
        "metrics_path": "runtime/model_selection/card/short_volshock_bc_checkpoint_alignment_v1/warm_start_official_metrics.csv",
    },
    {
        "source_label": "short_volshock_bc_initialization_official_replay_v1",
        "mechanism": "short_volshock_bc_initialization",
        "metrics_path": "runtime/model_selection/card/short_volshock_bc_initialization_official_replay_v1/official_replay_metrics.csv",
    },
    {
        "source_label": "short_volshock_official_replay_screen_v1",
        "mechanism": "short_volshock_action_cap_overlay",
        "metrics_path": "runtime/model_selection/card/short_volshock_official_replay_screen_v1/official_replay_metrics.csv",
    },
    {
        "source_label": "short_volshock_sell_guard_official_replay_v1",
        "mechanism": "short_volshock_sell_guard_overlay",
        "metrics_path": "runtime/model_selection/card/short_volshock_sell_guard_official_replay_v1/official_replay_metrics.csv",
    },
    {
        "source_label": "official_replay_risk_state_target_screen_v1",
        "mechanism": "risk_state_target_replay",
        "metrics_path": "runtime/model_selection/card/official_replay_risk_state_target_screen_v1/official_replay_metrics.csv",
    },
]

retention_gate = load_json("runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json") or {}
retention_summary = summary_of(retention_gate)
handoff_summary = load_json("runtime/model_selection/card/walkforward_retention_next_training_handoff_v1/summary.json") or {}
strict_summary = summary_of(load_json("runtime/model_selection/card/strict_candidate_retention_filter_v1/summary.json") or {})
final_summary = summary_of(load_json("runtime/model_selection/card/final_walkforward_retention_contract_v1/summary.json") or {})

source_manifest_rows = []
for source in SOURCE_TABLE:
    path = ROOT / source["metrics_path"]
    base = dict(source)
    base["exists"] = path.exists()
    base["rows"] = int(len(pd.read_csv(path))) if path.exists() else 0
    selection_path = path.parent / "selection.json"
    summary_path = path.parent / "summary.json"
    base["selection_path"] = rel(selection_path) if selection_path.exists() else ""
    base["summary_path"] = rel(summary_path) if summary_path.exists() else ""
    selection_payload = load_json(rel(selection_path)) if selection_path.exists() else {}
    summary_payload = load_json(rel(summary_path)) if summary_path.exists() else {}
    base["source_decision"] = summary_of(selection_payload).get("decision") or summary_of(summary_payload).get("decision") or ""
    source_manifest_rows.append(base)

source_manifest = pd.DataFrame(source_manifest_rows)
source_manifest_path = write_csv("source_manifest.csv", source_manifest)
source_manifest_json_path = write_json("source_manifest.json", source_manifest_rows)
print(f"repo_root={ROOT}")
print(f"source_manifest={source_manifest_path}")
display(source_manifest)

repo_root=/Users/rziiiii/dev/workspace
source_manifest=runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1/source_manifest.csv


,source_label,mechanism,metrics_path,exists,rows,selection_path,summary_path,source_decision
0,sequence_trend_joint_warm_start_official_eval_v1,sequence_trend_joint_bc,runtime/model_selection/card/sequence_trend_jo...,True,6,runtime/model_selection/card/sequence_trend_jo...,runtime/model_selection/card/sequence_trend_jo...,warm_start_official_eval_no_full_gate_no_promo...
1,state_conditioned_multi_objective_bc_screen_v1,state_conditioned_multi_objective_bc,runtime/model_selection/card/state_conditioned...,True,6,runtime/model_selection/card/state_conditioned...,runtime/model_selection/card/state_conditioned...,state_conditioned_multi_objective_drawdown_fai...
2,short_volshock_bc_checkpoint_alignment_v1,short_volshock_checkpoint_bc,runtime/model_selection/card/short_volshock_bc...,True,6,runtime/model_selection/card/short_volshock_bc...,runtime/model_selection/card/short_volshock_bc...,short_volshock_bc_replay_gate_failed_no_promotion
3,short_volshock_bc_initialization_official_repl...,short_volshock_bc_initialization,runtime/model_selection/card/short_volshock_bc...,True,2,runtime/model_selection/card/short_volshock_bc...,runtime/model_selection/card/short_volshock_bc...,short_volshock_bc_initialization_official_repl...
4,short_volshock_official_replay_screen_v1,short_volshock_action_cap_overlay,runtime/model_selection/card/short_volshock_of...,True,8,runtime/model_selection/card/short_volshock_of...,runtime/model_selection/card/short_volshock_of...,official_short_volshock_cap_replay_failed_no_p...
5,short_volshock_sell_guard_official_replay_v1,short_volshock_sell_guard_overlay,runtime/model_selection/card/short_volshock_se...,True,8,runtime/model_selection/card/short_volshock_se...,runtime/model_selection/card/short_volshock_se...,official_short_volshock_sell_guard_no_improvem...
6,official_replay_risk_state_target_screen_v1,risk_state_target_replay,runtime/model_selection/card/official_replay_r...,True,10,runtime/model_selection/card/official_replay_r...,runtime/model_selection/card/official_replay_r...,risk_state_target_cap_reduces_drawdown_but_mis...


## Normalize Official Replay Candidates

Every source table is converted into the same fold-level schema: candidate id, source, mechanism, window, relative return, MaxDD, average position, and official window gate. The scoring contract uses `relative_return >= +30pp` and `MaxDD <= 15%` for each evaluated official warm-start window.

In [2]:
normalised_frames = []
load_errors = []
raw_shape_rows = []
for source in SOURCE_TABLE:
    path = ROOT / source["metrics_path"]
    if not path.exists():
        load_errors.append({"source_label": source["source_label"], "error": "missing_metrics_path"})
        continue
    frame = pd.read_csv(path)
    raw_shape_rows.append({
        "source_label": source["source_label"],
        "path": source["metrics_path"],
        "rows": int(len(frame)),
        "columns": ",".join(frame.columns),
    })
    try:
        normalised = normalise_official_replay_candidate_frame(
            frame,
            source_label=source["source_label"],
            mechanism=source["mechanism"],
            config=CONFIG,
        )
    except Exception as exc:
        load_errors.append({"source_label": source["source_label"], "error": f"{type(exc).__name__}: {exc}"})
        continue
    normalised_frames.append(normalised)

candidate_pool = pd.concat(normalised_frames, ignore_index=True) if normalised_frames else pd.DataFrame()
raw_shapes = pd.DataFrame(raw_shape_rows)
load_error_df = pd.DataFrame(load_errors)

candidate_pool_path = write_csv("normalised_official_candidate_pool.csv", candidate_pool)
raw_shapes_path = write_csv("raw_source_shapes.csv", raw_shapes)
load_errors_path = write_csv("load_errors.csv", load_error_df) if not load_error_df.empty else ""
print(f"normalised_official_candidate_pool={candidate_pool_path}")
print(f"raw_source_shapes={raw_shapes_path}")
if load_errors_path:
    print(f"load_errors={load_errors_path}")

display(candidate_pool.head(20))

normalised_official_candidate_pool=runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1/normalised_official_candidate_pool.csv
raw_source_shapes=runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1/raw_source_shapes.csv


,source_label,mechanism,candidate,candidate_id,window,relative_return,max_dd,avg_pos,window_gate_pass,relative_gap,max_dd_gap,robust_window_gate
0,sequence_trend_joint_warm_start_official_eval_v1,sequence_trend_joint_bc,base_plain_contract,sequence_trend_joint_warm_start_official_eval_...,wf_2022,67.898818,20.198957,0.131701,False,0.000000,5.198957,False
1,sequence_trend_joint_warm_start_official_eval_v1,sequence_trend_joint_bc,base_plain_contract,sequence_trend_joint_warm_start_official_eval_...,wf_2023_2024,-0.136944,20.987807,0.159047,False,30.136944,5.987807,False
2,sequence_trend_joint_warm_start_official_eval_v1,sequence_trend_joint_bc,trend_plain_contract,sequence_trend_joint_warm_start_official_eval_...,wf_2022,67.898818,20.198957,0.131701,False,0.000000,5.198957,False
3,sequence_trend_joint_warm_start_official_eval_v1,sequence_trend_joint_bc,trend_plain_contract,sequence_trend_joint_warm_start_official_eval_...,wf_2023_2024,8.224563,12.285728,0.147325,False,21.775437,0.000000,False
4,sequence_trend_joint_warm_start_official_eval_v1,sequence_trend_joint_bc,trend_plus_slope_low_cap15_weighted,sequence_trend_joint_warm_start_official_eval_...,wf_2022,58.730532,28.944800,0.134772,False,0.000000,13.944800,False
5,sequence_trend_joint_warm_start_official_eval_v1,sequence_trend_joint_bc,trend_plus_slope_low_cap15_weighted,sequence_trend_joint_warm_start_official_eval_...,wf_2023_2024,6.334730,12.528079,0.140492,False,23.665270,0.000000,False
6,state_conditioned_multi_objective_bc_screen_v1,state_conditioned_multi_objective_bc,multi_obj_floor45_cap20_riskfirst,state_conditioned_multi_objective_bc_screen_v1...,wf_2022,59.388029,30.626799,0.164765,False,0.000000,15.626799,False
7,state_conditioned_multi_objective_bc_screen_v1,state_conditioned_multi_objective_bc,multi_obj_floor45_cap20_riskfirst,state_conditioned_multi_objective_bc_screen_v1...,wf_2023_2024,30.262521,16.122029,0.221759,False,0.000000,1.122029,False
8,state_conditioned_multi_objective_bc_screen_v1,state_conditioned_multi_objective_bc,multi_obj_floor55_cap25_balanced,state_conditioned_multi_objective_bc_screen_v1...,wf_2022,55.825533,31.808525,0.162252,False,0.000000,16.808525,False
9,state_conditioned_multi_objective_bc_screen_v1,state_conditioned_multi_objective_bc,multi_obj_floor55_cap25_balanced,state_conditioned_multi_objective_bc_screen_v1...,wf_2023_2024,8.057465,19.036155,0.209318,False,21.942535,4.036155,False


## Candidate Pool Score

The primary score is conservative: full gate requires every evaluated warm-start window to pass. Near-miss means all but one evaluated window pass and the remaining miss is within `1.0pp` relative return and `0.5%` MaxDD. Near-miss remains diagnostic only.

In [3]:
scored = score_official_replay_candidate_pool(candidate_pool, config=CONFIG)

coverage_rows = []
if not candidate_pool.empty:
    for candidate_id, sub in candidate_pool.groupby("candidate_id", sort=True):
        windows = set(sub["window"].astype(str))
        coverage_rows.append({
            "candidate_id": candidate_id,
            "source_label": str(sub["source_label"].iloc[0]),
            "mechanism": str(sub["mechanism"].iloc[0]),
            "candidate": str(sub["candidate"].iloc[0]),
            "present_windows": ";".join(sorted(windows)),
            "missing_eval_windows": ";".join(sorted(EVAL_REQUIRED_WINDOWS - windows)),
            "missing_final_required_windows": ";".join(sorted(FINAL_REQUIRED_WINDOWS - windows)),
            "eval_window_count": int(len(windows & EVAL_REQUIRED_WINDOWS)),
            "final_window_count": int(len(windows & FINAL_REQUIRED_WINDOWS)),
            "final_window_coverage_gate": FINAL_REQUIRED_WINDOWS.issubset(windows),
        })
coverage = pd.DataFrame(coverage_rows)
scored_with_coverage = scored.merge(
    coverage[["candidate_id", "present_windows", "missing_eval_windows", "missing_final_required_windows", "final_window_coverage_gate"]],
    on="candidate_id",
    how="left",
) if not scored.empty else scored
if not scored_with_coverage.empty:
    scored_with_coverage["final_retention_ready"] = (
        scored_with_coverage["full_gate"].astype(bool)
        & scored_with_coverage["final_window_coverage_gate"].fillna(False).astype(bool)
        & (final_summary.get("final_retained_count", 0) > 0)
    )
else:
    scored_with_coverage["final_retention_ready"] = []

scored_path = write_csv("scored_official_candidate_pool.csv", scored_with_coverage)
coverage_path = write_csv("candidate_window_coverage.csv", coverage)
review_text = render_official_replay_candidate_review(scored)
(CARD / "official_candidate_review.md").write_text(review_text)
review_path = rel(CARD / "official_candidate_review.md")
print(f"scored_official_candidate_pool={scored_path}")
print(f"candidate_window_coverage={coverage_path}")
print(f"official_candidate_review={review_path}")
display(scored_with_coverage.head(25))

scored_official_candidate_pool=runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1/scored_official_candidate_pool.csv
candidate_window_coverage=runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1/candidate_window_coverage.csv
official_candidate_review=runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1/official_candidate_review.md


,candidate_id,source_label,mechanism,candidate,window_count,gate_pass_count,full_gate,robust_gate_pass_count,robust_full_gate,near_miss,...,max_dd_gap,min_relative_margin,min_max_dd_margin,total_gate_loss,failure_profile,present_windows,missing_eval_windows,missing_final_required_windows,final_window_coverage_gate,final_retention_ready
0,short_volshock_official_replay_screen_v1::shor...,short_volshock_official_replay_screen_v1,short_volshock_action_cap_overlay,short_volshock_cap05_hit2,2,1,False,1,False,True,...,0.107381,-0.182737,-0.107381,1.256545,near_miss,wf_2022;wf_2023_2024,,wf_2020,False,False
1,short_volshock_sell_guard_official_replay_v1::...,short_volshock_sell_guard_official_replay_v1,short_volshock_sell_guard_overlay,short_volshock_cap05_hit2_no_guard,2,1,False,1,False,True,...,0.107381,-0.182737,-0.107381,1.256545,near_miss,wf_2022;wf_2023_2024,,wf_2020,False,False
2,short_volshock_official_replay_screen_v1::shor...,short_volshock_official_replay_screen_v1,short_volshock_action_cap_overlay,short_volshock_cap05_hit1,2,1,False,1,False,False,...,0.000000,-2.899001,1.167535,2.899001,trend_capture,wf_2022;wf_2023_2024,,wf_2020,False,False
3,short_volshock_sell_guard_official_replay_v1::...,short_volshock_sell_guard_official_replay_v1,short_volshock_sell_guard_overlay,short_volshock_cap05_hit2_sell_guard_zero,2,1,False,1,False,False,...,0.515531,2.213744,-0.515531,5.155312,drawdown_control,wf_2022;wf_2023_2024,,wf_2020,False,False
4,short_volshock_sell_guard_official_replay_v1::...,short_volshock_sell_guard_official_replay_v1,short_volshock_sell_guard_overlay,short_volshock_cap05_hit2_sell_guard_1pct,2,1,False,1,False,False,...,0.517909,2.320004,-0.517909,5.179090,drawdown_control,wf_2022;wf_2023_2024,,wf_2020,False,False
5,short_volshock_official_replay_screen_v1::shor...,short_volshock_official_replay_screen_v1,short_volshock_action_cap_overlay,short_volshock_cap08_hit1,2,1,False,0,False,False,...,0.520370,-6.383042,-0.520370,11.586743,joint_tradeoff,wf_2022;wf_2023_2024,,wf_2020,False,False
6,official_replay_risk_state_target_screen_v1::v...,official_replay_risk_state_target_screen_v1,risk_state_target_replay,volstress_all_targets_cap010,2,1,False,1,False,False,...,0.111239,-27.997940,-0.111239,29.110326,trend_capture,wf_2022;wf_2023_2024,,wf_2020,False,False
7,official_replay_risk_state_target_screen_v1::v...,official_replay_risk_state_target_screen_v1,risk_state_target_replay,volstress_all_targets_cap005,2,1,False,1,False,False,...,1.184661,-33.160067,-1.184661,45.006679,joint_tradeoff,wf_2022;wf_2023_2024,,wf_2020,False,False
8,sequence_trend_joint_warm_start_official_eval_...,sequence_trend_joint_warm_start_official_eval_v1,sequence_trend_joint_bc,trend_plain_contract,2,0,False,0,False,False,...,5.198957,-21.775437,-5.198957,73.765009,joint_tradeoff,wf_2022;wf_2023_2024,,wf_2020,False,False
9,official_replay_risk_state_target_screen_v1::b...,official_replay_risk_state_target_screen_v1,risk_state_target_replay,bc_initialization,2,0,False,0,False,False,...,7.261989,-24.981101,-7.261989,114.531866,joint_tradeoff,wf_2022;wf_2023_2024,,wf_2020,False,False


## Stress Scenarios

Stress is a promotion-blocking check on already-generated official replay metrics. It does not replace a true replay with changed execution semantics.

In [4]:
STRESS_SCENARIOS = [
    {"scenario": "base", "relative_haircut_pp": 0.0, "max_dd_add_pct": 0.0},
    {"scenario": "mild_cost_slippage", "relative_haircut_pp": 0.25, "max_dd_add_pct": 0.10},
    {"scenario": "one_point_relative_half_dd", "relative_haircut_pp": 1.0, "max_dd_add_pct": 0.50},
]
stress_scores = score_official_replay_stress_scenarios(
    candidate_pool,
    stress_scenarios=STRESS_SCENARIOS,
    config=CONFIG,
)
stress_path = write_csv("stress_scores.csv", stress_scores)
print(f"stress_scores={stress_path}")
display(stress_scores.head(40))

stress_scores=runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1/stress_scores.csv


,candidate_id,source_label,mechanism,candidate,scenario,relative_haircut_pp,max_dd_add_pct,window_count,stress_gate_pass_count,stress_full_gate,stress_min_relative_return,stress_max_dd,stress_total_gate_loss,stress_failure_profile
0,short_volshock_official_replay_screen_v1::shor...,short_volshock_official_replay_screen_v1,short_volshock_action_cap_overlay,short_volshock_cap05_hit2,base,0.00,0.0,2,1,False,29.817263,15.107381,1.256545,stress_near_miss
1,short_volshock_sell_guard_official_replay_v1::...,short_volshock_sell_guard_official_replay_v1,short_volshock_sell_guard_overlay,short_volshock_cap05_hit2_no_guard,base,0.00,0.0,2,1,False,29.817263,15.107381,1.256545,stress_near_miss
2,short_volshock_official_replay_screen_v1::shor...,short_volshock_official_replay_screen_v1,short_volshock_action_cap_overlay,short_volshock_cap05_hit1,base,0.00,0.0,2,1,False,27.100999,13.832465,2.899001,stress_near_miss
3,short_volshock_sell_guard_official_replay_v1::...,short_volshock_sell_guard_official_replay_v1,short_volshock_sell_guard_overlay,short_volshock_cap05_hit2_sell_guard_zero,base,0.00,0.0,2,1,False,32.213744,15.515531,5.155312,stress_near_miss
4,short_volshock_sell_guard_official_replay_v1::...,short_volshock_sell_guard_official_replay_v1,short_volshock_sell_guard_overlay,short_volshock_cap05_hit2_sell_guard_1pct,base,0.00,0.0,2,1,False,32.320004,15.517909,5.179090,stress_near_miss
5,short_volshock_official_replay_screen_v1::shor...,short_volshock_official_replay_screen_v1,short_volshock_action_cap_overlay,short_volshock_cap08_hit1,base,0.00,0.0,2,1,False,23.616958,15.520370,11.586743,stress_near_miss
6,official_replay_risk_state_target_screen_v1::v...,official_replay_risk_state_target_screen_v1,risk_state_target_replay,volstress_all_targets_cap010,base,0.00,0.0,2,1,False,2.002060,15.111239,29.110326,stress_near_miss
7,official_replay_risk_state_target_screen_v1::v...,official_replay_risk_state_target_screen_v1,risk_state_target_replay,volstress_all_targets_cap005,base,0.00,0.0,2,1,False,-3.160067,16.184661,45.006679,stress_near_miss
8,sequence_trend_joint_warm_start_official_eval_...,sequence_trend_joint_warm_start_official_eval_v1,sequence_trend_joint_bc,trend_plain_contract,base,0.00,0.0,2,0,False,8.224563,20.198957,73.765009,stress_joint_tradeoff
9,official_replay_risk_state_target_screen_v1::b...,official_replay_risk_state_target_screen_v1,risk_state_target_replay,bc_initialization,base,0.00,0.0,2,0,False,5.018899,22.261989,114.531866,stress_joint_tradeoff


## Selection And Handoff

This selection is deliberately stricter than simply sorting by loss. A candidate can be selected as a teacher/reference only if it is the best official replay candidate and its blockers are explicit. Promotion requires full final retention coverage, which is not available from these two-window BC screens.

In [5]:
if scored_with_coverage.empty:
    selected = {}
    selected_fold_metrics = pd.DataFrame()
    decision = "official_replay_metric_aligned_checkpoint_bc_no_candidates"
    next_mechanism = "repair_official_replay_metric_inputs_before_training"
else:
    selected_row = scored_with_coverage.iloc[0].to_dict()
    selected = selected_row
    selected_fold_metrics = candidate_pool[candidate_pool["candidate_id"] == selected_row["candidate_id"]].copy()
    if bool(selected_row.get("final_retention_ready", False)):
        decision = "official_replay_metric_aligned_checkpoint_bc_final_retention_ready"
        next_mechanism = "run_full_strict_retention_confirmation_before_actor_retention"
    elif bool(selected_row.get("full_gate", False)):
        decision = "official_replay_metric_aligned_checkpoint_bc_full_eval_gate_no_final_retention"
        next_mechanism = "add_wf2020_and_current_ytd_strict_retention_replay_before_actor_retention"
    elif bool(selected_row.get("near_miss", False)):
        decision = "official_replay_metric_aligned_checkpoint_bc_near_miss_no_promotion"
        next_mechanism = "use_short_volshock_near_miss_as_metric_aligned_checkpoint_or_bc_teacher_then_add_wf2020_and_strict_retention_gate"
    else:
        decision = "official_replay_metric_aligned_checkpoint_bc_no_promotion"
        next_mechanism = "regenerate_fold_local_joint_targets_or_add_new_state_feature_before_training"

selected_fold_path = write_csv("selected_candidate_fold_metrics.csv", selected_fold_metrics)
selection = {
    "experiment": "official_replay_metric_aligned_checkpoint_bc_screen_v1",
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "decision": decision,
    "selected_candidate_id": selected.get("candidate_id"),
    "selected_source_label": selected.get("source_label"),
    "selected_mechanism": selected.get("mechanism"),
    "selected_candidate": selected.get("candidate"),
    "selected_full_gate": bool(selected.get("full_gate", False)),
    "selected_near_miss": bool(selected.get("near_miss", False)),
    "selected_failure_profile": selected.get("failure_profile"),
    "selected_total_gate_loss": selected.get("total_gate_loss"),
    "selected_min_relative_return": selected.get("min_relative_return"),
    "selected_max_max_dd": selected.get("max_max_dd"),
    "selected_gate_pass_count": selected.get("gate_pass_count"),
    "selected_window_count": selected.get("window_count"),
    "selected_missing_final_required_windows": selected.get("missing_final_required_windows"),
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "ppo_allowed_next": False,
    "checkpoint_bc_reference_allowed_next": bool(selected.get("near_miss", False) or selected.get("full_gate", False)),
    "next_mechanism": next_mechanism,
    "hard_blockers": [
        "no candidate has final retained actor status",
        "BC official replay screens do not include all final required windows",
        "strict retention gate still has passing_profile_count=0",
        "current best-return alignment and MaxDD<20 final gate remain unproven for any new actor",
    ],
    "required_final_windows": sorted(FINAL_REQUIRED_WINDOWS),
    "official_eval_windows": sorted(EVAL_REQUIRED_WINDOWS),
    "strict_retention_context": {
        "strict_passing_profile_count": strict_summary.get("passing_profile_count"),
        "final_retained_count": final_summary.get("final_retained_count"),
        "required_best_return_pct": retention_summary.get("required_best_return_pct"),
        "max_walkforward_drawdown_pct": retention_summary.get("max_walkforward_drawdown_pct"),
        "reference_profile": retention_summary.get("reference_profile"),
        "reference_return_pct": retention_summary.get("reference_return_pct"),
        "current_best_return_pct": retention_summary.get("current_best_return_pct"),
    },
    "source_handoff_decision": handoff_summary.get("decision"),
    "sidecars": {
        "source_manifest": source_manifest_path,
        "source_manifest_json": source_manifest_json_path,
        "raw_source_shapes": raw_shapes_path,
        "load_errors": load_errors_path,
        "normalised_official_candidate_pool": candidate_pool_path,
        "scored_official_candidate_pool": scored_path,
        "candidate_window_coverage": coverage_path,
        "stress_scores": stress_path,
        "selected_candidate_fold_metrics": selected_fold_path,
        "official_candidate_review": review_path,
        "selection": "runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1/selection.json",
        "summary": "runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1/summary.json",
    },
}
selection_path = write_json("selection.json", selection)
print(json.dumps(selection, indent=2, sort_keys=True))
print(f"selection={selection_path}")
display(selected_fold_metrics)

{
  "actor_retention_allowed": false,
  "checkpoint_bc_reference_allowed_next": true,
  "created_at": "2026-06-16T03:08:20+00:00",
  "decision": "official_replay_metric_aligned_checkpoint_bc_near_miss_no_promotion",
  "experiment": "official_replay_metric_aligned_checkpoint_bc_screen_v1",
  "hard_blockers": [
    "no candidate has final retained actor status",
    "BC official replay screens do not include all final required windows",
    "strict retention gate still has passing_profile_count=0",
    "current best-return alignment and MaxDD<20 final gate remain unproven for any new actor"
  ],
  "next_mechanism": "use_short_volshock_near_miss_as_metric_aligned_checkpoint_or_bc_teacher_then_add_wf2020_and_strict_retention_gate",
  "official_eval_windows": [
    "wf_2022",
    "wf_2023_2024"
  ],
  "ppo_allowed_next": false,
  "promotion_allowed": false,
  "required_final_windows": [
    "wf_2020",
    "wf_2022",
    "wf_2023_2024"
  ],
  "selected_candidate": "short_volshock_cap05_hit2"

,source_label,mechanism,candidate,candidate_id,window,relative_return,max_dd,avg_pos,window_gate_pass,relative_gap,max_dd_gap,robust_window_gate
24,short_volshock_official_replay_screen_v1,short_volshock_action_cap_overlay,short_volshock_cap05_hit2,short_volshock_official_replay_screen_v1::shor...,wf_2022,83.564183,8.746809,0.062850,True,0.000000,0.000000,True
25,short_volshock_official_replay_screen_v1,short_volshock_action_cap_overlay,short_volshock_cap05_hit2,short_volshock_official_replay_screen_v1::shor...,wf_2023_2024,29.817263,15.107381,0.207768,False,0.182737,0.107381,False


## Summary Artifact

The summary keeps the selected near-miss visible while preserving the no-promotion decision. The next training step should consume this as a metric-aligned teacher/checkpoint reference, not as an actor.

In [6]:
if not scored_with_coverage.empty:
    near_miss_count = int(scored_with_coverage["near_miss"].fillna(False).astype(bool).sum())
    full_gate_count = int(scored_with_coverage["full_gate"].fillna(False).astype(bool).sum())
    final_ready_count = int(scored_with_coverage["final_retention_ready"].fillna(False).astype(bool).sum())
else:
    near_miss_count = full_gate_count = final_ready_count = 0

summary = {
    "experiment": "official_replay_metric_aligned_checkpoint_bc_screen_v1",
    "created_at": selection["created_at"],
    "decision": selection["decision"],
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "ppo_allowed_next": False,
    "checkpoint_bc_reference_allowed_next": selection["checkpoint_bc_reference_allowed_next"],
    "candidate_source_count": int(len(source_manifest)),
    "candidate_pool_rows": int(len(candidate_pool)),
    "scored_candidate_count": int(len(scored_with_coverage)),
    "full_gate_count": full_gate_count,
    "near_miss_count": near_miss_count,
    "final_retention_ready_count": final_ready_count,
    "selected": {
        key: selection.get(key)
        for key in [
            "selected_candidate_id",
            "selected_source_label",
            "selected_mechanism",
            "selected_candidate",
            "selected_full_gate",
            "selected_near_miss",
            "selected_failure_profile",
            "selected_total_gate_loss",
            "selected_min_relative_return",
            "selected_max_max_dd",
            "selected_missing_final_required_windows",
        ]
    },
    "next_mechanism": selection["next_mechanism"],
    "do_not_run": [
        "do_not_promote_near_miss_candidate",
        "do_not_run_longer_ppo_from_source_only_or_cold_split_metrics",
        "do_not_repeat_threshold_only_sell_guard_or_action_cap_grid",
        "do_not_treat_soxlbacktest_reference_baseline_as_actor",
    ],
    "next_action_queue": [
        {
            "priority": 1,
            "action_id": "short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1",
            "reason": "best official replay candidate is a near-miss; use its official daily path and loss profile as the checkpoint/BC objective reference",
            "required_gates": [
                "official daily replay metrics only",
                "wf_2020 coverage added before actor retention",
                "strict retention return alignment >= current best threshold",
                "walk-forward MaxDD < 20% final gate",
            ],
        },
        {
            "priority": 2,
            "action_id": "regenerate_fold_local_joint_reward_targets_v2",
            "reason": "if teacher/checkpoint objective cannot fix the near-miss without leaking, repair target scope before training",
            "required_gates": ["fold-local scope", "eval_overlap_rows == 0", "BC slice fit before PPO"],
        },
        {
            "priority": 3,
            "action_id": "tradingview_dashboard_interaction_spec_v1",
            "reason": "front-end interaction work remains separate; use current data freshness metadata and model-difference panes as the contract",
            "required_gates": ["no pointer lag", "crosshair/range state sync", "selected model date/readout visible"],
        },
    ],
    "strict_retention_context": selection["strict_retention_context"],
    "sidecars": selection["sidecars"],
}
summary_path = write_json("summary.json", summary)
print(json.dumps(summary, indent=2, sort_keys=True))
print(f"summary={summary_path}")

{
  "actor_retention_allowed": false,
  "candidate_pool_rows": 46,
  "candidate_source_count": 7,
  "checkpoint_bc_reference_allowed_next": true,
  "created_at": "2026-06-16T03:08:20+00:00",
  "decision": "official_replay_metric_aligned_checkpoint_bc_near_miss_no_promotion",
  "do_not_run": [
    "do_not_promote_near_miss_candidate",
    "do_not_run_longer_ppo_from_source_only_or_cold_split_metrics",
    "do_not_repeat_threshold_only_sell_guard_or_action_cap_grid",
    "do_not_treat_soxlbacktest_reference_baseline_as_actor"
  ],
  "experiment": "official_replay_metric_aligned_checkpoint_bc_screen_v1",
  "final_retention_ready_count": 0,
  "full_gate_count": 0,
  "near_miss_count": 2,
  "next_action_queue": [
    {
      "action_id": "short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1",
      "priority": 1,
      "reason": "best official replay candidate is a near-miss; use its official daily path and loss profile as the checkpoint/BC objective reference",
      "required_gates":

In [7]:
conclusion_md = f"""
## Conclusion

Decision: `{selection['decision']}`.

No candidate is promotable. Aggregated official replay candidates: `{summary['scored_candidate_count']}`; full official two-window gate candidates: `{summary['full_gate_count']}`; near-miss candidates: `{summary['near_miss_count']}`; final-retention-ready candidates: `{summary['final_retention_ready_count']}`.

Selected diagnostic reference: `{selection.get('selected_candidate_id')}` from `{selection.get('selected_mechanism')}`. It is selected because it is the lowest-loss official replay near-miss, not because it can be retained. Its fold metrics have min relative return `{selection.get('selected_min_relative_return'):.3f}pp` and max MaxDD `{selection.get('selected_max_max_dd'):.3f}%`; missing final required windows: `{selection.get('selected_missing_final_required_windows')}`.

The next model step should use the selected short-volshock near-miss as a metric-aligned checkpoint/BC teacher reference, add `wf_2020` coverage and strict current-YTD retention checks, and keep actor retention blocked until the final gate is proven. Dashboard interaction work remains a separate front-end track; it should consume the same freshness and model-difference metadata instead of implying that baseline is an updated actor.
""".strip()
(CARD / "conclusion.md").write_text(conclusion_md + "\n")
print(rel(CARD / "conclusion.md"))
display(Markdown(conclusion_md))

runtime/model_selection/card/official_replay_metric_aligned_checkpoint_bc_screen_v1/conclusion.md


## Conclusion

Decision: `official_replay_metric_aligned_checkpoint_bc_near_miss_no_promotion`.

No candidate is promotable. Aggregated official replay candidates: `23`; full official two-window gate candidates: `0`; near-miss candidates: `2`; final-retention-ready candidates: `0`.

Selected diagnostic reference: `short_volshock_official_replay_screen_v1::short_volshock_cap05_hit2` from `short_volshock_action_cap_overlay`. It is selected because it is the lowest-loss official replay near-miss, not because it can be retained. Its fold metrics have min relative return `29.817pp` and max MaxDD `15.107%`; missing final required windows: `wf_2020`.

The next model step should use the selected short-volshock near-miss as a metric-aligned checkpoint/BC teacher reference, add `wf_2020` coverage and strict current-YTD retention checks, and keep actor retention blocked until the final gate is proven. Dashboard interaction work remains a separate front-end track; it should consume the same freshness and model-difference metadata instead of implying that baseline is an updated actor.